In [0]:
%pip install sodapy==2.2.0

In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Execution Environment")
dbutils.widgets.dropdown("source", "yellow_taxi", ["yellow_taxi", "green_taxi"], "NYCOpenData")

env = dbutils.widgets.get('env')
source = dbutils.widgets.get('source')

print(f"{'='*70}")
print(f"Starting the process for {env} environment and {source} dataset")
print(f"{'='*70}")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

from sodapy import Socrata

from typing import Iterable
import time
import logging

In [0]:
# Retrieving the Secrets

scope          = 'db-scope'
aws_account_id = dbutils.secrets.get(scope, 'aws_account_id')
nyc_app_token  = dbutils.secrets.get(scope, 'nyc_app_token')

In [0]:
class NYCOpenData:

    def __init__(self, source: str, dataset_id: str, datetime_col: str):
        self.source       = source
        self.dataset_id   = dataset_id
        self.datetime_col = datetime_col

In [0]:
# Dataset config mapping
NYC_CONFIGS = {
    'yellow_taxi': {
        'dataset_id': '4b4i-vvec', 
        'datetime_col': 'tpep_pickup_datetime'
    },
    'green_taxi': {
        'dataset_id': 'peyi-gg4n', 
        'datetime_col': 'lpep_pickup_datetime'
    }
}

config = NYC_CONFIGS.get(source)

if config:
    data = NYCOpenData(source=source, **config)
else:
    raise ValueError(f"Data source {source} not recognised.")

In [0]:
def get_start_time(full_table_name: str) -> str:

    if not spark.catalog.tableExists(full_table_name):

        return '2023-01-01'

    return spark.read.table(full_table_name).agg(max("date_partition")).collect()[0][0]

In [0]:
# Execution variables

env             = "dev"
region          = "us-east-2"
layer           = "bronze"
project         = "ifood-data-architect-case"
s3_path         = f"{project}-{env}-{region}-{aws_account_id}-{layer}"
catalog         = "logistics"
schema          = layer
table_name      = data.source
delta_path      = f"s3://{s3_path}/{table_name}/"
full_table_name = f"{catalog}.{schema}.{table_name}"

# Ingestion variables
end_time   = '2023-06-01'
chunk_size = 50000

In [0]:
def chunk_generator(generator, chunk_size: int):

    chunk = []

    for item in generator:
        chunk.append(item)

        if not len(chunk) < chunk_size:
            yield chunk
            chunk = []
    
    if chunk:
        yield chunk

In [0]:

def get_data(data: NYCOpenData, start_time: str, token: str) -> Iterable[dict]:

  client = Socrata("data.cityofnewyork.us", token, timeout=120)
  date_filter = f"{data.datetime_col} >= '{start_time}T00:00:00' AND {data.datetime_col} < '{end_time}T00:00:00'"

  data_generator = client.get_all(
    dataset_identifier    = data.dataset_id,
    exclude_system_fields = False,
    where                 = date_filter
  )

  return data_generator

In [0]:
def data_prepare(df: DataFrame) -> DataFrame:

    prepared = (
                df
                .withColumn("id", col(":id"))
                .withColumn("pickup_datetime", col("tpep_pickup_datetime"))
                .withColumn("dropoff_datetime", col("tpep_dropoff_datetime"))
                .withColumn("_ingestion_at", current_timestamp())
                .withColumn("date_partition", date_format(col("tpep_pickup_datetime"), "yyyy-MM-dd"))
                .selectExpr(
                    'id',
                    'dolocationid',
                    'extra',
                    'fare_amount',
                    'improvement_surcharge',
                    'mta_tax',
                    'payment_type',
                    'pulocationid',
                    'tip_amount',
                    'tolls_amount',
                    'total_amount',
                    'pickup_datetime',
                    'dropoff_datetime',
                    'trip_distance',
                    'vendorid',
                    'airport_fee',
                    'congestion_surcharge',
                    'passenger_count',
                    'ratecodeid',
                    'store_and_fwd_flag',
                    '_ingestion_at',
                    'date_partition'
                )
    )

    return prepared

In [0]:
def merge_chunk(df: DataFrame, delta_path: str, full_table_name: str):

    if not spark.catalog.tableExists(full_table_name):
        
        (
            df
            .write
            .format("delta")
            .mode("overwrite")
            .option("path", delta_path)
            .partitionBy("date_partition")
            .saveAsTable(full_table_name)
        )

        spark.sql(f"ALTER TABLE {full_table_name} ALTER COLUMN id SET NOT NULL")
        spark.sql(f"ALTER TABLE {full_table_name} ADD CONSTRAINT id_unique CHECK(id IS NOT NULL)")
        spark.sql(f"ALTER TABLE {full_table_name} ADD CONSTRAINT taxi_pk PRIMARY KEY (id)")

    delta_table = DeltaTable.forName(spark, full_table_name)

    (
        delta_table.alias("target")
        .merge(
            source    = df.dropDuplicates(["id"]).alias("source"),
            condition = "target.id = source.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def ingest_chunks(data: Iterable[dict], delta_path: str, full_table_name: str, chunk_size: int):

    chunk_index = 1
    max_retries = 3

    print("Chunk ingestion has started.")
    print(f"{'-'*70}\n")

    for chunk in chunk_generator(data, chunk_size):
        attempt = 0
        success = False

        while attempt < max_retries and not success:

            try:

                df = spark.createDataFrame(chunk)
                bronze_prepared = data_prepare(df)
                merge_chunk(bronze_prepared, delta_path, full_table_name)
                print(f"Chunk #{chunk_index} ingested")

                chunk_index += 1
                success = True
            except Exception as e:
                attempt += 1
                wait_time = attempt * 30

                logging.error(f"Error ingesting chunk #{chunk_index} (Attempt {attempt}/{max_retries}): {e}")

                if attempt < max_retries:
                    print(f"Waiting {wait_time}s before trying it again.")
                    time.sleep(wait_time)
                else: 
                    print(f"Chunk #{chunk_index} failed after {max_retries} trials.")
                    raise e

    
    print(f"Completed the ingestion of {chunk_index - 1} chunks.")

In [0]:
def main(data: NYCOpenData, token: str, delta_path: str, full_table_name: str, chunk_size: int) -> None:
    max_job_retries = 5
    job_attempt = 0
    success = False

    while job_attempt < max_job_retries and not success:
        try:
            start_time = get_start_time(full_table_name)
            print(f"Starting ingestion from: {start_time}")
            
            data_generator = get_data(data, start_time, token)
            ingest_chunks(data_generator, delta_path, full_table_name, chunk_size)
            
            success = True
        except Exception as e:
            job_attempt += 1
            print(f"Critic fail (Attempt {job_attempt}/{max_job_retries}): {e}")
            if job_attempt < max_job_retries:
                time.sleep(60)
            else:
                raise e

In [0]:
main(
    data,
    nyc_app_token,
    delta_path,
    full_table_name,
    chunk_size
)